# Viscosity Coefficients

## Prelude
In this notebook we will calculate the viscosity of the Yukawa OCP.

The YAML input file can be found at [input_file](https://raw.githubusercontent.com/murillo-group/sarkas/master/docs/examples/YOCP/input_files/yocp_transport.yaml) and this notebook at [notebook](https://raw.githubusercontent.com/murillo-group/sarkas/master/docs/examples/YOCP/YOCP_Transport_NB.ipynb).


In [ ]:
# Import the usual libraries
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import os
plt.style.use('MSUstyle')

# Import sarkas
from sarkas.processes import Simulation, PostProcess, PreProcess
# import observables
from sarkas.tools.observables import Thermodynamics, PressureTensor, HeatFlux, RadialDistributionFunction, VelocityAutoCorrelationFunction



In [ ]:
# Create the file path to the YAML input file
input_file_name = os.path.join('input_files', 'yocp_adaptivetherm.yaml')

sim = Simulation(input_file_name)
sim.setup(read_yaml=True)

In [ ]:
sim.adaptive_thermalize()

In [ ]:
# sim.timer.stop()
sim.produce()

In [ ]:
sim.parameters.equilibration_steps

In [ ]:
postproc = PostProcess(input_file_name)
postproc.setup(read_yaml=True)
# postproc.parameters.verbose = True

## Thermodynamics

Let's take a look at the temperature and energy of the system

In [ ]:
therm = Thermodynamics()
therm.setup(postproc.parameters, phase="equilibration")
therm.compute()
therm.temp_energy_plot(postproc)

In [ ]:
therm.calculate_beta_slices()

### Key Observations

#### Temperature

- **Temperature Behavior**:
  - During the production phase (bottom left plot), the temperature fluctuates around the desired value of 11.60 kK.
  - The moving average of the temperature (orange line) closely follows the desired temperature (red dashed line), indicating that the system maintained the desired temperature during the equilibration phase and carried this stability into the production phase.
  
- **Fluctuations**:
  - The temperature fluctuations are within an acceptable range, suggesting that the system is well-equilibrated and stable during the NVE production phase.

#### Energy

- **Total Energy Fluctuations**:
  - The total energy plot (bottom right) shows fluctuations around a mean value.
  - The moving average of the total energy (orange line) remains relatively constant, indicating energy conservation in the NVE ensemble.

- **Energy Distribution**:
  - The histogram of the energy values shows a roughly Gaussian distribution, typical for an equilibrated system in an NVE ensemble.
  - The standard deviation of the energy distribution is small, indicating stable energy fluctuations around the mean.

### System Stability and Equilibration

- **System Stability**:
  - Both the temperature and energy plots indicate that the system has reached a stable state after initial equilibration. The thermostat used in the $NVT$ equilibration phase successfully prepared the system for stable dynamics in the $NVE$ production phase.
  
- **Thermostat Efficiency**:
  - The Berendsen thermostat effectively maintained the temperature during the equilibration phase, allowing the system to transition smoothly into the $NVE$ production phase with stable temperature and energy fluctuations.

- **Fluctuation Analysis**:
  - The magnitude of temperature and energy fluctuations appears to be small, indicating a well-equilibrated system with minimal drift during the $NVE$ production phase.

### Conclusion

The provided plot shows that the MD simulation resulted in a stable and well-equilibrated system. The temperature and energy remained stable throughout the production phase, confirming that the system maintained the desired conditions set during the equilibration phase.

## Pair Distribution Function

The first observable to calculate is always the RDF. 

First, we initialize the RDF computation by creating an instance of the `RadialDistributionFunction` class. We then set up the RDF calculation parameters using the setup method and compute the RDF using the compute method.

In [ ]:
rdf = RadialDistributionFunction()
rdf.setup(postproc.parameters)
rdf.compute()

After running the RDF computation, Sarkas provides a detailed summary of the calculation parameters and results.

Finally, we plot the computed RDF using the provided plotting functionality. We normalize the x-axis by the characteristic length scale $a_{ws}$ and label the axes appropriately.

In [ ]:
_ = rdf.plot(
    scaling = rdf.a_ws, 
    y = ('Al-Al RDF', 'Mean'),
    xlabel = r'$r /a_{ws}$',
    ylabel = r"$g(r)$",
    legend = False
)

The plot shows the radial distribution function $g(r)$ as a function of the normalized distance $r/a_{ws}$. This function describes how particle density varies as a function of distance from a reference particle, providing insights into the local structure of the system.

## Pressure Tensor

The viscosity is obtained from the autocorrelation function (ACF) of the Pressure Tensor $\overleftrightarrow{\mathcal P}$ whose elements are

\begin{equation}
\mathcal P_{\alpha\gamma}(t) = \frac{1}{V} \sum_{i}^{N} \left [ m_i v^{\alpha}_{i} v^{\gamma}_{i} -  \sum_{j > i} \frac{r_{ij}^{\alpha} r_{ij}^{\gamma} }{r_{ij}} \frac{d}{dr}\phi(r) \right ],
\end{equation}

where $r_{ij}^{\alpha}$ is the $\alpha$ component of the distance between particles $i$ and $j$. The first term is the kinetic term and the second term is the virial term, but it is often referred to as the potential contribution. The virial is calculated during the simulation phase and saved together with particles corrdinates. 

Let's calculate the Pressure tensor and the pressure $\mathcal P$.

In [ ]:
# from sarkas.tools.observables import PressureTensor

In [ ]:
pt = PressureTensor()
pt.setup(postproc.parameters)
pt.compute()

As usual the data is saved in several dataframes. In this case we have 4 dataframes

* A dataframe for the values of each of the elements of the pressure tensor for each of the slices, `pt.dataframe_slices`
* A dataframe for the mean and std values of each of the elements of the pressure tensor, `pt.dataframe`
* A dataframe for the ACF of each pair $\langle \mathcal P_{\alpha\beta}(t)\mathcal P_{\mu\nu}(0) \rangle$ for each slice, `pt.dataframe_acf_slices`
* A dataframe for the mean and std of the ACF of each pair $\langle \mathcal P_{\alpha\beta}(t)\mathcal P_{\mu\nu}(0) \rangle$, `pt.dataframe_acf`

The above code snipper computed only the pressure tensor elements and not their autocorrelations. Let's look at `pt.dataframe` and its columns

In [ ]:
pt.dataframe.info()

In [ ]:
pt.dataframe

There are 15 columns in this dataset; the time column, a `Mean` and `Std` column for scalar pressure $\mathcal P(t)$ `(Total, Pressure, Mean/Std)`, and `Mean` and `Std` column for elements of the tensor. Note that only the elements (`XY`,`XZ`, `YZ`) are calculated since the tensor is supposed to be symmetric in the isotropic case. 

Let's plot the Pressure as a function of time

In [ ]:
# Let's plot it
t_wp = pt.plasma_period

p_id = pt.total_num_density / therm.beta_slices.mean()
ax = pt.plot( 
    scaling = (t_wp, p_id),
    y = ("Total","Pressure", "Mean"),
    xlabel = "Plasma cycles",
    ylabel = r"$ \beta P(t)/n$"
       )
ax.plot(
    pt.dataframe[("Total", "Quantity", 'Time')]/t_wp,
    pt.dataframe[("Total",'Pressure','Mean')].expanding().mean()/p_id )
ax.legend(['Pressure', 'Cumulative Avg'])

This plot shows that the pressure is constant throughout the simulation. It would be nice to have some more information though. In fact we can make plots showing the deviation from the average pressure and a histogram of the pressure. The following code shows us how.

In [ ]:
from sarkas.tools.observables import make_gaussian_plot

In [ ]:
fig, axes = make_gaussian_plot(
    time = pt.dataframe[("Total", "Quantity", 'Time')]/t_wp,
    data = pt.dataframe[("Total",'Pressure Tensor XY','Mean')],
    xlabel = 'Plasma periods',
    ylabel = r"$ \beta P_{ex}(t)/n$"
)

This figure is useful to understand the properties of our data. The main plot shows the (normalized) pressure as a function of time rescaled by the plasma period. The dashed red line indicates the mean of the time series and the orange line is the cumulative average. The top plot shows the standard deviation of our data as a function of time with the corresponding cumulative average. The plot on the right, finally, shows a histogram of the pressure data (green), while the orange line is Gaussian distribution whose mean and standard deviation are obtained from the pressure data.

Things to be understood from this plot is that the pressure is a stationary process, meaning that its mean and std do not change with time. In more technical term, the pressure is a stationary process if the data does not have a unit-root. While the plot is a good visual there are statistical tests to confirm the stationarity of time series data. These are the ADF and KPSS test. 

The code below runs these statistical tests and ouputs their results.

In [ ]:
from sarkas.tools.observables import check_stationarity

check_stationarity(pt.dataframe[("Total", "Pressure", "Mean")]/p_id)

## Pressure from RDF

In order to check that our code are correct, let's verify some laws. 

The pressure of the system is calculated from $\mathcal P(t)= \frac1{3} {\rm Tr} \overleftrightarrow{\mathcal P}(t)$ and also from 

\begin{equation}
P = n k_BT - \frac{4\pi}{6} n^2 \int_0^{\infty} dr \, r^3 g(r) \frac{d\phi(r)}{dr} 
\end{equation}

where $g(r)= 1 + h(r)$ is the pair distribution function that we have already calculated. In YOCP case we can calculate the Hartree term and obtain

\begin{equation}
P = n k_BT + \frac{3}{2} n k_BT \frac{\Gamma}{\kappa^2} - \frac{4\pi}{6} n^2 \int_0^{\infty} dr \, r^3 h(r) \frac{d\phi(r)}{dr} 
\end{equation}

Let's now calculate the pressure from the integral of the RDF. This is obtained from the method `compute_from_rdf` of the `Thermodynamics` object. 

Looking at the documentation of this [method](../../api/tools_subpckg/Thermodynamics_methods/sarkas.tools.observables.Thermodynamics.compute_from_rdf.rst) we notice that it returns five values:
the Hartree and correlational terms between species $A$ and $B$ and the ideal pressure $n k_BT$. 

The total pressure is given from the sum of the three terms and should be equal to the 

$$ P = n k_BT + P_{\rm Hartree} + P_{\rm Corr} = {\operatorname {Mean} } \left \{ \mathcal P(t) \right \} $$

In [ ]:
nkT, _, _, p_h, p_c = therm.compute_from_rdf(rdf, postproc.potential)
kappa = postproc.potential.a_ws / postproc.potential.screening_length
p_h_yocp = postproc.potential.coupling_constant * 1.5 / kappa**2
print(f"The ratio between the calculated Hartree term and the theoretical value is: {p_h[0]/nkT / p_h_yocp :.4e}")

P_rdf = nkT + p_h + p_c
P_trace = pt.dataframe[("Total","Pressure", "Mean")].mean()

rel_diff = (P_rdf[0] - P_trace)*100/P_rdf[0] 
print(f"The relative difference between the integral of g(r) and the mean of the time series data is = {rel_diff:.2f} %")

It seems that we have done a good job! Let's move on and calculate the autocorrelation functions (ACFs). 
We can do this by calling the `compute_acf` method.

In [ ]:
pt.compute_acf()

In [ ]:
pt.dataframe_acf.info()

In [ ]:
pt.dataframe_acf

### Sum rule

Let's now check that we have calculated the ACF correctly. The equal time ACFs of the elements of $\overleftrightarrow{\mathcal P}(t)$ obey the following sum rules

$$
\mathcal J_{zzzz}(0) = \frac 13 \sum_{\alpha}\left \langle \mathcal P_{\alpha\alpha}(0)\mathcal P_{\alpha\alpha}(0) \right \rangle  =  \frac{n}{\beta^2} \left [ 3 + \frac{2\beta}{15} I_1 + \frac \beta5 I_2 \right ] ,
$$ 
$$
\mathcal J_{zzxx}(0) = \frac 16 \sum_{\alpha} \sum_{\beta\neq\alpha} \left \langle \mathcal P_{\alpha\alpha}(0)\mathcal P_{\beta\beta}(0) \right \rangle = \frac{n}{\beta^2} \left [ 1 - \frac{2\beta}{5} I_1 + \frac \beta{15} I_2 \right ] ,
$$ 
$$
\mathcal J_{xyxy}(0) = \frac 16 \sum_{\alpha}\sum_{\beta \neq \alpha} \left \langle \mathcal P_{\alpha\beta}(0)\mathcal P_{\alpha\beta}(0) \right \rangle = \frac{n}{\beta^2} \left [ 1 + \frac{4\beta}{15} I_1 + \frac \beta{15} I_2 \right ] ,
$$ 

where

$$ 
I_1 = 2\pi n \int dr \, r^3 g(r)  \frac{d\phi(r)}{dr}, \quad I_2 = 2\pi n \int dr\, r^3 g(r) \frac{d^2\phi(r)}{dr^2}.
$$

Notice that all three equal time ACF satisfy 

$$ \mathcal J_{zzzz}(0) - \mathcal J_{zzxx}(0) = 2 \mathcal J_{xyxy}(0) .$$

In [ ]:
# Diagonal terms
column_zzzz = [
    ('Pressure Tensor ACF XXXX', 'Mean'),
    ('Pressure Tensor ACF YYYY', 'Mean'),
    ('Pressure Tensor ACF ZZZZ', 'Mean'),
]
J_zzzz_0 = pt.dataframe_acf[column_zzzz].iloc[0].mean()
    
# Cross-Diagonal Terms
column_zzxx = [
    ('Pressure Tensor ACF XXYY', 'Mean'),
    ('Pressure Tensor ACF XXZZ', 'Mean'),
    ('Pressure Tensor ACF YYZZ', 'Mean')
]
J_zzxx_0 = pt.dataframe_acf[column_zzxx].iloc[0].mean()
    
# Cross Off Diagonal terms
column_xyxy = [
    ('Pressure Tensor ACF XYXY', 'Mean'),
    ('Pressure Tensor ACF XZXZ', 'Mean'),
    ('Pressure Tensor ACF YZYZ', 'Mean')
]
J_xyxy_0 = pt.dataframe_acf[column_xyxy].iloc[0].mean()

# The units of J's are [Density *  Energy]^2
condition = (J_zzzz_0 - J_zzxx_0)/(2.0 * J_xyxy_0)
msg = f'The isotropy condition : (J_zzzz_0 - J_zzxx_0 )/( 2*J_xyxy_0 ) = {condition:.4f}'
print(msg)

cross_off = pt.dataframe_acf[column_xyxy].mean(axis = 1)
cross_off /= cross_off.iloc[0]

t_wp = 2.0*np.pi/ pt.total_plasma_frequency
ax = pt.plot(scaling = t_wp, 
             y = column_xyxy,
             acf = True,
             xlabel = "Plasma periods",
             ls = '--',
             label = [r"$\langle P_{xy}(\tau)P_{xy}(0)\rangle$ ACF", r"$\langle P_{xz}(\tau)P_{xz}(0)\rangle$ ACF", r"$\langle P_{yz}(\tau)P_{yz}(0)\rangle$ ACF"]
         )
ax.set(xscale = 'symlog')
time_col = pt.dataframe_acf.columns[0]
ax.plot( pt.dataframe_acf[time_col]/t_wp, cross_off, ls = ':', label = "Average")
_ = ax.legend()
# fig = ax.figure
# fig.savefig( os.path.join(pic_fldr,"Pxy_ACF_plot.png") )

Let's now verify the sum rules. These are calculated from the `pt.sum_rule` method which takes as inputs the inverse temperature $\beta$, the `RadialDistributionFunction` instance, and the `Potential` instance. 

In [ ]:
# Calculate the elastic constants from the integrals of g(r)
sigma_zzzz, sigma_zzxx, sigma_xyxy = pt.sum_rule(therm.beta_slices.mean(), rdf, postproc.potential)
print(f"{sigma_zzzz:.3e}, {sigma_zzxx:.3e}, {sigma_xyxy:.3e}")

In [ ]:
# Shear modulus
G_inf = J_xyxy_0*therm.beta_slices.mean()*pt.box_volume
K_inf = 1.0/3.0*(J_zzzz_0 + 2.0* J_zzxx_0)*therm.beta_slices.mean()*pt.box_volume

K_sr = (sigma_zzzz + 2.0*sigma_zzxx)/3.0

print("G_inf = {:.3e}, sum_rule = {:.3e}, ratio = {:.4f}".format( G_inf, sigma_xyxy, abs(sigma_xyxy) /G_inf) )
print("K_inf = {:.3e}, sum_rule = {:.3e}, ratio = {:.2f}".format( K_inf,  K_sr, K_sr /K_inf) )

In [ ]:
from scipy.integrate import cumulative_trapezoid

acfs = (pt.dataframe_acf[column_xyxy]/pt.dataframe_acf[column_xyxy].iloc[0]).values
corr_times = 2.0 * cumulative_trapezoid(acfs**2, x = pt.dataframe_acf.iloc[:,0] / pt.plasma_period, axis = 0)
fig, ax = plt.subplots(1,1)
ax.plot(pt.dataframe_acf.iloc[:-1,0]/pt.plasma_period,corr_times[:,0], label = 'XY')
ax.plot(pt.dataframe_acf.iloc[:-1,0]/pt.plasma_period,corr_times[:,1], label = 'XZ')
ax.plot(pt.dataframe_acf.iloc[:-1,0]/pt.plasma_period,corr_times[:,2], label = 'YZ')
ax.set(xlabel = 'Plasma Periods', ylabel = 'Correlation time', xscale = 'symlog', ylim = (0, 2))

In [ ]:
acf_df = pt.calc_better_acf_data(plasma_periods_shift = 100)
acf_df.info()

In [ ]:
acf_df

In [ ]:
# Plot
acf_0 = acf_df["Stress ACF Mean"].iloc[0]
fig, ax = plt.subplots(1,1)
ax.plot(acf_df["Time"], acf_df["Stress ACF Mean"]/acf_0)
ax.fill_between(acf_df["Time"], 
                acf_df["Stress ACF Mean"]/acf_0 - acf_df["Stress ACF Std"]/acf_0,
               acf_df["Stress ACF Mean"]/acf_0 + acf_df["Stress ACF Std"]/acf_0,
               alpha = 0.3)
ax.set(xscale= 'log', ylim = (-0.1, 1.1))

## Viscosity

The shear viscosity is calculated from the Green-Kubo relation

\begin{equation}
\eta = \frac{\beta V}{3} \sum_{\alpha} \sum_{\gamma \neq \alpha} \int_0^{\infty} dt \, \left \langle \delta\mathcal P_{\alpha\gamma}(t) \delta \mathcal P_{\alpha\gamma}(0) \right \rangle,
\end{equation}

where $\beta = 1/k_B T$, $\alpha,\gamma = {x, y, z}$ and $\delta \mathcal P_{\alpha\gamma}(t) = \mathcal P_{\alpha\gamma}(t) - \left \langle \mathcal P_{\alpha\gamma} \right \rangle_t$.

The bulk viscosity is given by a similar relation

\begin{equation}
\eta_V = \beta V \int_0^{\infty}dt \,  \left \langle \delta \mathcal P(t) \delta \mathcal P(0) \right \rangle,
\end{equation}

where $\delta \mathcal P(t) = \mathcal P(t) - \left \langle \mathcal P  \right \rangle_t$ is the fluctuation of the scalar pressure.

In [ ]:
from scipy.integrate import cumulative_trapezoid

eta_t = therm.beta_slices.mean() * pt.box_volume * cumulative_trapezoid(acf_df.iloc[:,1], x = acf_df.iloc[:, 0])

In [ ]:
def murillo_yvm(kappa, gamma):
    gamma_m_k = 171.8 + 82.8*(np.exp(0.565*kappa**1.38) - 1)
    eta_E = 0.0051 * gamma_m_k/gamma + 0.374 * gamma/gamma_m_k + 0.022
    eta = eta_E * omega_E/rdf.total_plasma_frequency # np.exp(-0.2 * kappa**1.62)/np.sqrt(3)
    return eta

def murillo_iyvm(kappa, gamma):
    gamma_m_k = 171.8 + 82.8*(np.exp(0.565*kappa**1.38) - 1)
    Ak = 1.45e-4 - 1.04e-4*kappa +3.69e-5*kappa**2
    ak = 1.78+0.13*kappa - 0.062*kappa**2
    Bk = 0.3+0.86*kappa - 0.69*kappa**2 +0.138 * kappa**3
    bk = 1.63 - 0.325 * kappa + 0.24*kappa**2
    Ck = 0.015 + 0.048 * kappa**0.754
    
    eta_E = Ak * (gamma_m_k/gamma)**ak + Bk * (gamma/gamma_m_k)**bk + Ck
    eta = eta_E * omega_E/rdf.total_plasma_frequency # np.exp(-0.2 * kappa**1.62)/np.sqrt(3)
    return eta

### Einstein frequency

The formula is 

\begin{equation}
\omega_E^2 = \frac{4\pi n}{3 m}\left [ 2\int_0^{\infty} dr \, r g(r) \frac{d\phi(r)}{dr}  + \int_0^{\infty} dr \, r^2 g(r) \frac{d^2\phi(r)}{dr^2} \right ] 
\end{equation}

In [ ]:
r = rdf.ra_values * rdf.a_ws
r[0] = 1e-40
_, du_dr, d2u_dr2 = postproc.potential.potential_derivatives(r, postproc.potential.matrix[0,0])
gr = rdf.dataframe[("Al-Al RDF", "Mean")].values
omega_E = np.sqrt(4.0 * np.pi * rdf.total_num_density/ (3 * rdf.species_masses[0]) * np.trapz( 2 * r * gr * du_dr  + r**2 * gr * d2u_dr2, x = r))
omega_E/rdf.total_plasma_frequency, np.exp(-0.2 * kappa**1.62)/ np.sqrt(3)

In [ ]:
time_rescaled = acf_df.iloc[1:,0] / pt.plasma_period
eta_rescale_const = pt.total_plasma_frequency * pt.a_ws**2 * pt.species_masses[0] * pt.total_num_density

kappa = pt.a_ws/postproc.potential.screening_length

fig, ax = plt.subplots(1,1)
ax.plot(time_rescaled,
        eta_t/eta_rescale_const,
       label = r'$\eta$')
eta_yvm = murillo_yvm(kappa, postproc.potential.coupling_constant)
eta_iyvm = murillo_iyvm(kappa, postproc.potential.coupling_constant)
ax.axhline(eta_yvm, ls = '--', c = 'r', label = "YVM")
ax.axhline(eta_iyvm, ls = ':', c = 'b', label = "IYVM")
ax.legend()
ax.set(xlabel = r'Plasma periods lag',
      ylabel = r"Shear viscosity $\eta$",
      xscale=  'log',
        ylim = (0, 0.05)
      )


## Thermal Conductivity

In [ ]:
from sarkas.tools.observables import HeatFlux
from sarkas.tools.transport import ThermalConductivity

In [ ]:
ht = HeatFlux()
ht.setup(postproc.parameters)
ht.compute(calculate_acf=True)

In [ ]:
ht.plot(scaling = t_wp,
        y = [("Heat Flux", 'Al', 'X', "Mean"),
             # ("Heat Flux", 'Al', 'Y', "Mean"), 
             # ("Heat Flux", 'Al', 'Z', "Mean")
            ],
       xlabel = 'Plasma Periods')

In [ ]:
thc = ThermalConductivity()
thc.setup(postproc.parameters, ht, therm)
thc.compute(ht)